# Chapter 8 Lab: Benchmarking QML Honestly
## Applied Quantum Computing — Optional Advanced Lab

**Objective:** Compare classical SVM (RBF kernel) with a quantum kernel SVM using rigorous statistical methodology: 5-seed cross-validation, paired t-test, and practical significance assessment.

**Dataset:** UCI Breast Cancer Wisconsin (569 samples, 30 features)

**Time:** ~3 hours

**Deliverables:**
1. All cells executed with results visible
2. Completed results table
3. 600-word vendor evaluation memo

## Setup: Install Dependencies

Run this cell first. Takes ~3 minutes on Colab.

In [ ]:
# Install required packages
!pip install qiskit==1.0.2 qiskit-machine-learning==0.7.2 scikit-learn numpy scipy matplotlib --quiet
print('Installation complete.')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score

from scipy import stats

from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel

print('All imports successful.')
print('Qiskit and Qiskit Machine Learning are ready.')

## Section 1: Load and Explore the Dataset

In [ ]:
# Load UCI Breast Cancer Wisconsin dataset
data = load_breast_cancer()
X, y = data.data, data.target

print(f'Dataset: {data.target_names}')
print(f'Samples: {X.shape[0]}, Features: {X.shape[1]}')
print(f'Class distribution — Malignant (0): {(y==0).sum()}, Benign (1): {(y==1).sum()}')

# Quick feature overview
df = pd.DataFrame(X, columns=data.feature_names)
df['target'] = y
df.head()

## Section 2: Classical SVM — Full Hyperparameter Tuning

**Key principle:** The classical baseline must be *maximally optimized* — not just competent. This means joint grid search over C and γ, cross-validated.

In [ ]:
# Configuration
SEEDS = [0, 1, 2, 3, 4]
TEST_SIZE = 0.2

# Hyperparameter grid — joint search over C and gamma
param_grid = {
    'svm__C': [0.1, 1, 10, 100, 1000],
    'svm__gamma': [1e-4, 1e-3, 0.01, 0.1, 1, 'scale']
}

classical_accuracies = []
classical_best_params = []

print('Training classical SVM with joint hyperparameter tuning...')
print('(5-fold CV grid search over 30 hyperparameter combinations x 5 seeds)\n')

for seed in SEEDS:
    # Stratified split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=seed, stratify=y
    )
    
    # Pipeline: scale → SVM
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(kernel='rbf'))
    ])
    
    # Grid search with 5-fold CV
    gs = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    gs.fit(X_train, y_train)
    
    acc = gs.score(X_test, y_test)
    classical_accuracies.append(acc)
    classical_best_params.append(gs.best_params_)
    
    print(f'Seed {seed}: Test Accuracy = {acc:.4f} | Best: {gs.best_params_}')

classical_mean = np.mean(classical_accuracies)
classical_std = np.std(classical_accuracies)
print(f'\nClassical SVM: Mean = {classical_mean:.4f} ± {classical_std:.4f}')

## Section 3: Quantum Kernel SVM

**Architecture:** ZZFeatureMap (6 qubits, 2 repetitions) + SVM

**Preprocessing:** PCA to 6 components (quantum circuit can only handle 6 features at this qubit count)

**Note:** This runs on a statevector simulator (no hardware noise). Results do not represent performance on real quantum hardware.

In [ ]:
import time

N_QUBITS = 6  # Must equal number of features after PCA
N_PCA_COMPONENTS = 6

# Build the quantum kernel
feature_map = ZZFeatureMap(feature_dimension=N_QUBITS, reps=2)
quantum_kernel = FidelityQuantumKernel(feature_map=feature_map)

print(f'Quantum feature map circuit depth: {feature_map.decompose().depth()}')
print(f'Qubits: {N_QUBITS}, Repetitions: 2')
print(f'PCA dimensions: {N_PCA_COMPONENTS}')
print('\nNote: Running on statevector simulator (noise-free). This is NOT hardware performance.\n')

quantum_accuracies = []
quantum_times = []

for seed in SEEDS:
    t0 = time.time()
    
    # Stratified split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=seed, stratify=y
    )
    
    # Scale and PCA reduce to 6 features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    pca = PCA(n_components=N_PCA_COMPONENTS, random_state=seed)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)
    
    # Normalize to [-1, 1] range for quantum circuit
    X_train_norm = X_train_pca / (np.max(np.abs(X_train_pca), axis=0) + 1e-8)
    X_test_norm = X_test_pca / (np.max(np.abs(X_train_pca), axis=0) + 1e-8)
    
    # Compute quantum kernel matrices
    print(f'Seed {seed}: Computing kernel matrices...')
    K_train = quantum_kernel.evaluate(x_vec=X_train_norm)
    K_test = quantum_kernel.evaluate(x_vec=X_test_norm, y_vec=X_train_norm)
    
    # Train SVM with precomputed kernel
    svm = SVC(kernel='precomputed', C=1.0)
    svm.fit(K_train, y_train)
    acc = svm.score(K_test, y_test)
    
    elapsed = time.time() - t0
    quantum_accuracies.append(acc)
    quantum_times.append(elapsed)
    
    print(f'  Accuracy = {acc:.4f} | Time = {elapsed:.1f}s')

quantum_mean = np.mean(quantum_accuracies)
quantum_std = np.std(quantum_accuracies)
print(f'\nQuantum Kernel SVM: Mean = {quantum_mean:.4f} ± {quantum_std:.4f}')
print(f'Average time per seed: {np.mean(quantum_times):.1f}s')

## Section 4: Statistical Analysis

**Paired t-test:** Tests whether the per-seed accuracy difference is statistically significant.

**Cohen's d:** Measures practical significance (effect size).

**Decision rule:** Quantum advantage is claimed ONLY if:
- p < 0.05 (statistical significance)
- Cohen's d > 0.2 (non-trivial effect size)
- Mean accuracy difference > 0.5 percentage points (practical significance)

In [ ]:
# Paired t-test
t_stat, p_value = stats.ttest_rel(quantum_accuracies, classical_accuracies)

# Cohen's d
diff = np.array(quantum_accuracies) - np.array(classical_accuracies)
cohens_d = diff.mean() / diff.std(ddof=1)

mean_diff = quantum_mean - classical_mean

print('=' * 60)
print('STATISTICAL ANALYSIS RESULTS')
print('=' * 60)
print(f'Classical SVM (tuned RBF):  {classical_mean:.4f} ± {classical_std:.4f}')
print(f'Quantum Kernel SVM:         {quantum_mean:.4f} ± {quantum_std:.4f}')
print(f'Mean difference (Q - C):    {mean_diff:+.4f}')
print(f'Paired t-statistic:         {t_stat:.4f}')
print(f'p-value:                    {p_value:.4f}')
print(f"Cohen's d:                  {cohens_d:.4f}")
print()
print('Decision Criteria:')
print(f'  p < 0.05?           {"YES" if p_value < 0.05 else "NO"} (p = {p_value:.4f})')
print(f"  Cohen's d > 0.2?    {\"YES\" if abs(cohens_d) > 0.2 else \"NO\"} (d = {cohens_d:.4f})")
print(f'  Diff > 0.5%?        {"YES" if abs(mean_diff) > 0.005 else "NO"} (diff = {mean_diff*100:.2f}%)')
print()

all_criteria = (p_value < 0.05) and (abs(cohens_d) > 0.2) and (abs(mean_diff) > 0.005)
if all_criteria and mean_diff > 0:
    verdict = 'QUANTUM ADVANTAGE CLAIM SUPPORTED (all three criteria met)'
elif all_criteria and mean_diff < 0:
    verdict = 'CLASSICAL ADVANTAGE (quantum performs worse — all criteria met for classical)'
else:
    verdict = 'NO QUANTUM ADVANTAGE DEMONSTRATED (at least one criterion not met)'

print(f'VERDICT: {verdict}')

In [ ]:
# Results Table
results = pd.DataFrame({
    'Seed': SEEDS,
    'Classical SVM Accuracy': [f'{a:.4f}' for a in classical_accuracies],
    'Quantum Kernel Accuracy': [f'{a:.4f}' for a in quantum_accuracies],
    'Difference (Q-C)': [f'{(q-c)*100:+.2f}%' for q, c in zip(quantum_accuracies, classical_accuracies)],
    'Quantum Time (s)': [f'{t:.1f}' for t in quantum_times]
})

summary = pd.DataFrame({
    'Seed': ['MEAN', 'STD'],
    'Classical SVM Accuracy': [f'{classical_mean:.4f}', f'{classical_std:.4f}'],
    'Quantum Kernel Accuracy': [f'{quantum_mean:.4f}', f'{quantum_std:.4f}'],
    'Difference (Q-C)': [f'{mean_diff*100:+.2f}%', '—'],
    'Quantum Time (s)': [f'{np.mean(quantum_times):.1f}', f'{np.std(quantum_times):.1f}']
})

full_table = pd.concat([results, summary], ignore_index=True)
print('RESULTS TABLE')
print('=' * 80)
print(full_table.to_string(index=False))
print()
print(f'Statistical test: paired t({len(SEEDS)-1}) = {t_stat:.3f}, p = {p_value:.4f}')
print(f"Effect size: Cohen's d = {cohens_d:.3f}")

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Per-seed comparison
ax = axes[0]
x = np.arange(len(SEEDS))
width = 0.35
bars1 = ax.bar(x - width/2, classical_accuracies, width, label='Classical SVM (tuned RBF)', 
               color='steelblue', alpha=0.8)
bars2 = ax.bar(x + width/2, quantum_accuracies, width, label='Quantum Kernel SVM',
               color='darkorange', alpha=0.8)
ax.axhline(y=classical_mean, color='steelblue', linestyle='--', alpha=0.5, label=f'Classical mean ({classical_mean:.3f})')
ax.axhline(y=quantum_mean, color='darkorange', linestyle='--', alpha=0.5, label=f'Quantum mean ({quantum_mean:.3f})')
ax.set_xlabel('Random Seed')
ax.set_ylabel('Test Accuracy')
ax.set_title('Per-Seed Accuracy Comparison')
ax.set_xticks(x)
ax.set_xticklabels([f'Seed {s}' for s in SEEDS])
ax.legend(fontsize=8)
ax.set_ylim(0.8, 1.02)
ax.grid(axis='y', alpha=0.3)

# Distribution comparison
ax2 = axes[1]
ax2.errorbar(['Classical SVM\n(tuned RBF)', 'Quantum Kernel\nSVM'], 
             [classical_mean, quantum_mean],
             yerr=[classical_std, quantum_std],
             fmt='o', markersize=10, capsize=8, capthick=2,
             color=['steelblue', 'darkorange'], ecolor=['steelblue', 'darkorange'])
ax2.set_ylabel('Test Accuracy (Mean ± Std)')
ax2.set_title(f'Mean Accuracy with Error Bars\n(p = {p_value:.4f}, d = {cohens_d:.3f})')
ax2.set_ylim(max(0.8, min(classical_mean, quantum_mean) - 0.05), 1.02)
ax2.grid(axis='y', alpha=0.3)

verdict_color = 'green' if all_criteria and mean_diff > 0 else 'red' if mean_diff < 0 else 'gray'
ax2.text(0.5, 0.02, 'QUANTUM ADVANTAGE' if (all_criteria and mean_diff > 0) else 'NO QUANTUM ADVANTAGE',
         transform=ax2.transAxes, ha='center', fontsize=9, color=verdict_color, weight='bold')

plt.suptitle('Chapter 8 Lab: QML Benchmark Results\nUCI Breast Cancer Wisconsin Dataset', 
             fontsize=12, weight='bold')
plt.tight_layout()
plt.savefig('ch08_qml_benchmark_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: ch08_qml_benchmark_results.png')

## Section 5: Vendor Evaluation Memo

**Instructions:** Replace the placeholder text below with your 600-word memo. The memo must address the five required topics listed.

---
# MEMO

**TO:** [Procurement Committee / Technical Review Board]

**FROM:** [Your Name]

**RE:** What I Would Tell a Vendor Who Claims Quantum Advantage on This Problem

**Date:** [Date]

---

## The Experiment and Its Results

*[Write 100–120 words summarizing what you found. Include the mean accuracy values, the statistical test result (p-value and Cohen's d), and a plain-English interpretation of whether quantum advantage was demonstrated.]*

## Statistical Significance

*[Write 100–120 words on what the statistical result means. If p > 0.05, explain why this matters. If p < 0.05 but Cohen's d is small, explain the distinction between statistical and practical significance. What sample size would be needed to detect a 0.5% accuracy difference with 80% power?]*

## Dataset and Feature Dimensionality Limitations

*[Write 100–120 words on the limitations of using this dataset for quantum advantage claims. Address: the 6-qubit circuit can only process 6 features (we used PCA to reduce from 30), so we are not testing quantum kernels on the full feature space. What does this mean for generalizability? How would the result differ if the quantum circuit could process all 30 features?]*

## The Simulator-Hardware Gap

*[Write 100–120 words on the simulator vs. hardware gap. This experiment ran on a noise-free statevector simulator. Real quantum hardware has gate errors of 0.1–1% per two-qubit gate, measurement errors, and decoherence. A ZZFeatureMap with 6 qubits and 2 repetitions has significant depth — what would hardware noise do to the kernel values? Cite this as a critical limitation if a vendor presents simulator-only results.]*

## My Recommendation on a Hardware Pilot

*[Write 100–120 words giving a clear recommendation: yes, no, or conditional. If conditional, specify what additional evidence (from the classical side and the quantum side) would be required before a hardware pilot could be justified. Reference the Classical Baseline Optimization requirement from Chapter 8. What minimum accuracy improvement over the best tuned classical baseline would justify the cost of a hardware pilot?]*

---

*Total word count: [count your words and insert here]*

## Reflection Questions

Answer these in the cell below before submitting.

1. The quantum kernel was tested on 6 PCA-reduced features. How would you expect performance to change if the quantum circuit had 30 qubits (matching the full feature set)?

2. The paired t-test compares per-seed accuracy values. What assumption does this test make about the relationship between seeds? Is that assumption valid here?

3. A vendor shows you this experiment's results (before you add the properly-tuned classical baseline) and claims quantum advantage. What is the specific number they are using to make that claim, and why is it misleading without the tuned baseline comparison?

*[Write your reflection answers here — replace this text]*

---

**Applied Quantum Computing | Chapter 8 Lab**

This notebook was developed for educational purposes. The quantum kernel experiments run on a statevector simulator (Qiskit). Results do not represent performance on real quantum hardware.

**Key references:**
- Havlíček et al. (2019). Supervised learning with quantum-enhanced feature spaces. *Nature*, 567, 209–212.
- Kübler et al. (2022). The inductive bias of quantum kernels. *NeurIPS 2021*.
- Tang, E. (2019). A quantum-inspired classical algorithm for recommendation systems. *STOC 2019*.
- Cerezo et al. (2021). Variational quantum algorithms. *Nature Reviews Physics*, 3, 625–644.